# Reto proyecto: Workflow con tres agentes usando LangChain

**Empresa ficticia:** EduFlowTech  
**Caso de uso:** automatización de tickets de soporte para una plataforma educativa online.

Este notebook implementa un flujo de trabajo con tres agentes funcionales:

1. **Agente 1 — Procesador de consultas:** interpreta la pregunta del usuario, detecta intención, tema, prioridad y tipo de consulta.
2. **Agente 2 — Buscador de contenido:** consulta una base de datos simulada con cursos, lecciones, ejercicios y metadatos.
3. **Agente 3 — Generador de respuestas:** construye una respuesta personalizada, clara y accionable.

El proyecto usa **LangChain Expression Language (LCEL)** mediante `RunnableLambda` para encadenar los agentes de forma modular. Además, incluye una base de datos ficticia con datos de texto, recursos visuales y metadatos.

## 1. Instalación de dependencias

Ejecuta esta celda en Google Colab o en tu entorno local si no tienes instaladas las librerías.

> Nota: se utiliza una API key ficticia para cumplir el requisito de seguridad. El workflow funciona en modo demo sin llamar a la API de OpenAI.

In [ ]:
# Descomenta esta línea si estás en Google Colab o si aún no tienes las dependencias instaladas.
# !pip install -q langchain langchain-core langchain-openai openai pandas

## 2. Imports y configuración

Se configura una clave ficticia de OpenAI. El notebook no realiza llamadas reales a la API por defecto, para que pueda ejecutarse sin coste y sin exponer credenciales.

In [ ]:
import os
import re
import json
import unicodedata
from datetime import datetime
from typing import Dict, List, Any

import pandas as pd
import openai

# API Key ficticia solicitada por el enunciado.
# Sustituir por una clave real solo si se quiere activar USE_OPENAI = True.
openai.api_key = "sk-YOUR_FAKE_API_KEY"
os.environ["OPENAI_API_KEY"] = "sk-YOUR_FAKE_API_KEY"

# LangChain: se usa LCEL para montar el workflow de agentes.
try:
    from langchain_core.runnables import RunnableLambda
    from langchain_core.prompts import PromptTemplate
    LANGCHAIN_AVAILABLE = True
except ImportError as exc:
    LANGCHAIN_AVAILABLE = False
    raise ImportError(
        "LangChain no está instalado. Ejecuta: pip install langchain langchain-core langchain-openai openai pandas"
    ) from exc

# Por defecto no se llama a OpenAI para que el proyecto sea reproducible sin API key real.
USE_OPENAI = False

print(f"LangChain disponible: {LANGCHAIN_AVAILABLE}")
print(f"Modo OpenAI real activado: {USE_OPENAI}")

## 3. Base de datos simulada de EduFlowTech

La base ficticia contiene cursos, lecciones, ejercicios, enlaces, recursos visuales y metadatos. Esto permite simular un entorno real donde los agentes consultan datos estructurados y devuelven respuestas personalizadas.

In [ ]:
content_data = [
    {
        "id": "DL-001",
        "curso": "Deep Learning Básico",
        "tema": "redes neuronales",
        "leccion": "Introducción a Redes Neuronales",
        "ejercicio": "Clasificación simple con una feed-forward network",
        "nivel": "principiante",
        "tipo_recurso": "texto + notebook",
        "descripcion": "Conceptos esenciales de neuronas, capas, pesos, bias, funciones de activación y entrenamiento básico.",
        "imagen_referencia": "https://eduflowtech.example/images/neural_network_intro.png",
        "link": "https://eduflowtech.example/cursos/deep-learning-basico/redes-neuronales",
        "duracion_min": 45,
        "prerrequisitos": "Python básico, conceptos de machine learning",
    },
    {
        "id": "DL-002",
        "curso": "Deep Learning Básico",
        "tema": "cnn computer vision imagenes",
        "leccion": "Redes Convolucionales para Computer Vision",
        "ejercicio": "Clasificación de imágenes con CNN",
        "nivel": "intermedio",
        "tipo_recurso": "texto + imágenes + notebook",
        "descripcion": "Uso de convoluciones, filtros, pooling y clasificación de imágenes con redes CNN.",
        "imagen_referencia": "https://eduflowtech.example/images/cnn_filters.png",
        "link": "https://eduflowtech.example/cursos/deep-learning-basico/cnn",
        "duracion_min": 60,
        "prerrequisitos": "Redes neuronales básicas, Python, manejo de imágenes",
    },
    {
        "id": "NLP-001",
        "curso": "Foundation Models en NLP",
        "tema": "llm transformers nlp tokenizacion",
        "leccion": "Modelos de Lenguaje Generativos y Tokenización",
        "ejercicio": "Comparación entre modelos instruction-tuned y no instruction-tuned",
        "nivel": "principiante",
        "tipo_recurso": "texto + notebook",
        "descripcion": "Introducción a LLMs, tokens, prompts, modelos generativos y diferencias con modelos no generativos.",
        "imagen_referencia": "https://eduflowtech.example/images/tokenization.png",
        "link": "https://eduflowtech.example/cursos/foundation-models-nlp/tokenizacion",
        "duracion_min": 55,
        "prerrequisitos": "Conceptos básicos de IA generativa",
    },
    {
        "id": "NLP-002",
        "curso": "Foundation Models en NLP",
        "tema": "zero-shot clasificacion emails soporte reclamacion comercial",
        "leccion": "Clasificación Zero-Shot de Textos",
        "ejercicio": "Clasificar emails de clientes sin dataset etiquetado",
        "nivel": "intermedio",
        "tipo_recurso": "texto + práctica guiada",
        "descripcion": "Uso de modelos de lenguaje para clasificar textos en categorías definidas sin entrenamiento supervisado.",
        "imagen_referencia": "https://eduflowtech.example/images/zero_shot_classification.png",
        "link": "https://eduflowtech.example/cursos/foundation-models-nlp/zero-shot",
        "duracion_min": 50,
        "prerrequisitos": "NLP básico, prompts",
    },
    {
        "id": "NLP-003",
        "curso": "Foundation Models en NLP",
        "tema": "fine-tuning rlhf temperatura creatividad prompt api openai",
        "leccion": "Ajuste de Modelos y Parámetros de Generación",
        "ejercicio": "Experimentación con temperatura y prompts creativos",
        "nivel": "avanzado",
        "tipo_recurso": "texto + notebook + API",
        "descripcion": "Explica fine-tuning, RLHF, temperatura, salidas generativas y configuración de llamadas a modelos.",
        "imagen_referencia": "https://eduflowtech.example/images/llm_parameters.png",
        "link": "https://eduflowtech.example/cursos/foundation-models-nlp/parametros-generacion",
        "duracion_min": 70,
        "prerrequisitos": "LLMs, Python, APIs",
    },
    {
        "id": "GEN-001",
        "curso": "IA Generativa con Diffusers",
        "tema": "stable diffusion difusion imagenes text-to-image image-to-image",
        "leccion": "Generación de Imágenes con Stable Diffusion",
        "ejercicio": "Crear imágenes con prompts y negative prompts",
        "nivel": "intermedio",
        "tipo_recurso": "notebook + imágenes",
        "descripcion": "Uso de modelos de difusión para generar imágenes y modificar imágenes existentes.",
        "imagen_referencia": "https://eduflowtech.example/images/stable_diffusion_grid.png",
        "link": "https://eduflowtech.example/cursos/ia-generativa-diffusers/stable-diffusion",
        "duracion_min": 65,
        "prerrequisitos": "Python, PyTorch básico, conceptos de IA generativa",
    },
    {
        "id": "ML-001",
        "curso": "Machine Learning Aplicado",
        "tema": "clasificacion regresion random forest svm arbol decision",
        "leccion": "Modelos Supervisados para Clasificación y Regresión",
        "ejercicio": "Predicción de estrés y clasificación de riesgo cardiovascular",
        "nivel": "principiante",
        "tipo_recurso": "texto + dataset tabular",
        "descripcion": "Árboles de decisión, random forest, SVM, regresión y evaluación de modelos supervisados.",
        "imagen_referencia": "https://eduflowtech.example/images/supervised_learning.png",
        "link": "https://eduflowtech.example/cursos/ml-aplicado/supervisado",
        "duracion_min": 75,
        "prerrequisitos": "Python, pandas, conceptos estadísticos básicos",
    },
    {
        "id": "ML-002",
        "curso": "Machine Learning Aplicado",
        "tema": "clustering pca no supervisado reduccion dimensiones",
        "leccion": "Aprendizaje No Supervisado: Clustering y PCA",
        "ejercicio": "Agrupación de pacientes y análisis de componentes principales",
        "nivel": "intermedio",
        "tipo_recurso": "texto + notebook + dataset tabular",
        "descripcion": "Técnicas para agrupar datos sin etiquetas y reducir dimensiones para análisis visual.",
        "imagen_referencia": "https://eduflowtech.example/images/clustering_pca.png",
        "link": "https://eduflowtech.example/cursos/ml-aplicado/no-supervisado",
        "duracion_min": 70,
        "prerrequisitos": "pandas, modelos supervisados básicos",
    },
    {
        "id": "SUP-001",
        "curso": "Soporte de Plataforma EduFlowTech",
        "tema": "bug error acceso notebook ejercicio colab api key instalacion",
        "leccion": "Guía de Resolución de Errores Frecuentes",
        "ejercicio": "Checklist para problemas de ejecución en notebooks",
        "nivel": "todos",
        "tipo_recurso": "guía + metadatos",
        "descripcion": "Solución de problemas habituales: dependencias, API keys, errores de importación, GPU y enlaces rotos.",
        "imagen_referencia": "https://eduflowtech.example/images/support_checklist.png",
        "link": "https://eduflowtech.example/soporte/errores-frecuentes",
        "duracion_min": 20,
        "prerrequisitos": "Ninguno",
    },
    {
        "id": "SUP-002",
        "curso": "Soporte de Plataforma EduFlowTech",
        "tema": "certificado progreso evaluacion entrega github repositorio",
        "leccion": "Entregas, GitHub y Evaluación Automática",
        "ejercicio": "Preparar un notebook para entrega en repositorio GitHub",
        "nivel": "todos",
        "tipo_recurso": "guía + checklist",
        "descripcion": "Buenas prácticas para entregar notebooks, organizar repositorios y comprobar requisitos de evaluación.",
        "imagen_referencia": "https://eduflowtech.example/images/github_submission.png",
        "link": "https://eduflowtech.example/soporte/entregas-github",
        "duracion_min": 25,
        "prerrequisitos": "Cuenta de GitHub",
    },
]

df_content = pd.DataFrame(content_data)

def normalize_text(text: str) -> str:
    # Normaliza texto: minúsculas, sin acentos y sin caracteres especiales complejos.
    text = str(text).lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = re.sub(r"[^a-z0-9ñ\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Guardamos también la base como CSV para simular un repositorio de datos.
os.makedirs("data", exist_ok=True)
df_content.to_csv("data/eduflowtech_content.csv", index=False)

display(df_content.head(10))

## 4. Diagrama del workflow

```mermaid
flowchart LR
    U[Usuario crea ticket] --> A1[Agente 1: Procesador de consultas]
    A1 -->|Categoría, intención, tema, prioridad| A2[Agente 2: Buscador de contenido]
    A2 -->|Lección, ejercicio, enlace, metadatos| A3[Agente 3: Generador de respuestas]
    A3 --> R[Respuesta personalizada al usuario]
    A2 --> DB[(Base de datos simulada: cursos, lecciones, ejercicios, imágenes, metadatos)]
```

## 5. Agente 1 — Procesador de consultas

Este agente recibe la consulta del usuario y devuelve una estructura con:

- categoría: lección, ejercicio, bug, entrega, consulta general;
- tema principal;
- términos extraídos;
- prioridad;
- si necesita revisar recursos visuales o metadatos.

In [ ]:
def process_query(user_query: str) -> Dict[str, Any]:
    # Agente 1: interpreta y categoriza la consulta del usuario.
    q_norm = normalize_text(user_query)

    categories = {
        "bug": ["error", "bug", "fallo", "no funciona", "no puedo", "problema", "api key", "import", "colab"],
        "entrega": ["github", "repositorio", "entrega", "subir", "evaluacion", "certificado", "notebook"],
        "ejercicio": ["ejercicio", "practica", "actividad", "reto", "codigo", "dataset"],
        "leccion": ["leccion", "aprender", "tema", "curso", "introduccion", "explicacion", "recomendada"],
        "media": ["imagen", "video", "captura", "grafico", "visual", "diagrama"],
    }

    scores = {cat: sum(1 for kw in keywords if kw in q_norm) for cat, keywords in categories.items()}
    category = max(scores, key=scores.get) if max(scores.values()) > 0 else "consulta_general"

    topic_keywords = [
        "redes neuronales", "deep learning", "cnn", "computer vision", "nlp", "llm", "transformers",
        "tokenizacion", "zero-shot", "clasificacion", "regresion", "clustering", "pca",
        "stable diffusion", "diffusers", "fine-tuning", "temperatura", "prompt", "github",
    ]
    extracted_terms = [kw for kw in topic_keywords if normalize_text(kw) in q_norm]
    topic = extracted_terms[0] if extracted_terms else "general"

    priority = "alta" if category == "bug" or any(word in q_norm for word in ["urgente", "bloqueado", "no puedo entregar"]) else "normal"
    needs_media = any(word in q_norm for word in ["imagen", "video", "captura", "visual", "diagrama"])

    return {
        "original_query": user_query,
        "normalized_query": q_norm,
        "category": category,
        "topic": topic,
        "extracted_terms": extracted_terms,
        "priority": priority,
        "needs_media": needs_media,
        "agent_1_summary": f"Consulta categorizada como '{category}' sobre '{topic}' con prioridad {priority}.",
    }

# Prueba rápida del Agente 1
process_query("¿Cuál es la lección más adecuada para aprender sobre redes neuronales?")

## 6. Agente 2 — Buscador de contenido

Este agente recibe la salida del Agente 1 y busca el recurso más relevante en la base simulada. Usa una puntuación sencilla basada en coincidencias semánticas por palabras clave, categoría y nivel.

In [ ]:
def score_record(processed_query: Dict[str, Any], record: pd.Series) -> int:
    # Calcula una puntuación simple para ordenar los contenidos más relevantes.
    q = processed_query["normalized_query"]
    category = processed_query["category"]
    extracted_terms = processed_query["extracted_terms"]

    searchable_text = normalize_text(
        " ".join([
            str(record["curso"]), str(record["tema"]), str(record["leccion"]),
            str(record["ejercicio"]), str(record["descripcion"]), str(record["tipo_recurso"]),
            str(record["prerrequisitos"]), str(record["nivel"]),
        ])
    )

    score = 0
    # Coincidencia por términos extraídos. Se prioriza que el tema aparezca en tema/lección,
    # no solo en prerrequisitos, para que la recomendación sea más precisa.
    primary_text = normalize_text(str(record["tema"]) + " " + str(record["leccion"]))
    for term in extracted_terms:
        term_norm = normalize_text(term)
        if term_norm in primary_text:
            score += 10
        elif term_norm in searchable_text:
            score += 4

    for token in set(q.split()):
        if len(token) > 3 and token in searchable_text:
            score += 1

    if category == "bug" and "soporte" in searchable_text:
        score += 8
    if category == "entrega" and ("github" in searchable_text or "entrega" in searchable_text):
        score += 8
    if category == "ejercicio" and "ejercicio" in searchable_text:
        score += 3
    if category == "leccion" and "leccion" in searchable_text:
        score += 3
    if category == "media" and any(x in searchable_text for x in ["imagenes", "imagen", "visual", "video"]):
        score += 3

    return score


def search_content(processed_query: Dict[str, Any]) -> Dict[str, Any]:
    # Agente 2: busca contenidos relevantes en la base de datos simulada.
    scored_rows = []
    for _, row in df_content.iterrows():
        score = score_record(processed_query, row)
        row_dict = row.to_dict()
        row_dict["score"] = score
        scored_rows.append(row_dict)

    ranked = sorted(scored_rows, key=lambda item: item["score"], reverse=True)
    top_matches = ranked[:3]
    best = top_matches[0]
    confidence = "alta" if best["score"] >= 8 else "media" if best["score"] >= 4 else "baja"

    return {
        **processed_query,
        "best_match": best,
        "top_matches": top_matches,
        "search_confidence": confidence,
        "agent_2_summary": f"Contenido principal encontrado: {best['leccion']} ({best['curso']}) con confianza {confidence}.",
    }

# Prueba rápida del Agente 2
search_content(process_query("¿Cuál es la lección más adecuada para aprender sobre redes neuronales?"))["best_match"]

## 7. Agente 3 — Generador de respuestas

Este agente toma el resultado de búsqueda y genera una respuesta personalizada. Incluye:

- recomendación principal;
- motivo de la elección;
- enlace simulado;
- ejercicio recomendado;
- tratamiento especial si la consulta es un bug o una entrega;
- referencia a recursos visuales y metadatos cuando procede.

In [ ]:
response_prompt_template = PromptTemplate.from_template(
    """
    Eres un agente de soporte de EduFlowTech.
    Debes responder de forma clara, breve y útil.

    Consulta original: {original_query}
    Categoría: {category}
    Prioridad: {priority}
    Curso recomendado: {curso}
    Lección recomendada: {leccion}
    Ejercicio recomendado: {ejercicio}
    Nivel: {nivel}
    Link: {link}
    Descripción: {descripcion}
    """
)


def generate_response(search_result: Dict[str, Any]) -> Dict[str, Any]:
    # Agente 3: genera una respuesta personalizada para el usuario.
    best = search_result["best_match"]
    category = search_result["category"]
    priority = search_result["priority"]

    llm_ready_prompt = response_prompt_template.format(
        original_query=search_result["original_query"],
        category=category,
        priority=priority,
        curso=best["curso"],
        leccion=best["leccion"],
        ejercicio=best["ejercicio"],
        nivel=best["nivel"],
        link=best["link"],
        descripcion=best["descripcion"],
    )

    if category == "bug":
        answer = (
            f"He detectado que tu consulta parece un problema técnico de prioridad {priority}. "
            f"Te recomiendo revisar la guía **'{best['leccion']}'** del curso **'{best['curso']}'**.\n\n"
            f"Pasos sugeridos:\n"
            f"1. Comprueba que las dependencias estén instaladas correctamente.\n"
            f"2. Verifica si estás usando la API key o el entorno adecuado.\n"
            f"3. Revisa el ejercicio **'{best['ejercicio']}'** como checklist de diagnóstico.\n"
            f"4. Si el error continúa, abre un ticket adjuntando captura y mensaje completo del error.\n\n"
            f"Enlace recomendado: {best['link']}"
        )
    elif category == "entrega":
        answer = (
            f"Para preparar correctamente tu entrega, revisa **'{best['leccion']}'**. "
            f"La recomendación principal es subir solo el notebook `.ipynb` cuando el enunciado lo pida, "
            f"manteniendo el repositorio ordenado por módulo o laboratorio.\n\n"
            f"Recurso recomendado: **{best['ejercicio']}**\n"
            f"Enlace: {best['link']}"
        )
    elif category == "ejercicio":
        answer = (
            f"El ejercicio más adecuado para tu consulta es **'{best['ejercicio']}'**, dentro de la lección "
            f"**'{best['leccion']}'** del curso **'{best['curso']}'**.\n\n"
            f"Nivel: {best['nivel']}\n"
            f"Por qué encaja: {best['descripcion']}\n"
            f"Enlace: {best['link']}"
        )
    else:
        answer = (
            f"La lección recomendada es **'{best['leccion']}'**, del curso **'{best['curso']}'**.\n\n"
            f"Motivo: {best['descripcion']}\n"
            f"Ejercicio sugerido: **{best['ejercicio']}**\n"
            f"Nivel: {best['nivel']}\n"
            f"Duración estimada: {best['duracion_min']} minutos\n"
            f"Enlace: {best['link']}"
        )

    if search_result["needs_media"] or "imagen" in normalize_text(best["tipo_recurso"]):
        answer += f"\n\nRecurso visual asociado: {best['imagen_referencia']}"

    return {
        "query": search_result["original_query"],
        "category": category,
        "priority": priority,
        "recommended_course": best["curso"],
        "recommended_lesson": best["leccion"],
        "recommended_exercise": best["ejercicio"],
        "confidence": search_result["search_confidence"],
        "answer": answer,
        "top_matches": [
            {"id": item["id"], "curso": item["curso"], "leccion": item["leccion"], "score": item["score"]}
            for item in search_result["top_matches"]
        ],
        "metadata": {
            "generated_at": datetime.now().isoformat(timespec="seconds"),
            "data_types_used": ["texto", "metadatos", "recurso_visual"],
            "workflow": "Agente 1 -> Agente 2 -> Agente 3",
            "llm_ready_prompt_preview": llm_ready_prompt[:500],
        },
    }

# Prueba rápida del Agente 3
generate_response(search_content(process_query("¿Cuál es la lección más adecuada para aprender sobre redes neuronales?")))["answer"]

## 8. Construcción del workflow con LangChain

Se encadenan los tres agentes mediante `RunnableLambda`. Esto permite que el output de un agente sea el input del siguiente de forma clara, modular y reutilizable.

In [ ]:
# Cada función se convierte en un componente ejecutable de LangChain.
query_processor_agent = RunnableLambda(process_query)
content_search_agent = RunnableLambda(search_content)
response_generator_agent = RunnableLambda(generate_response)

# Workflow completo: Agente 1 -> Agente 2 -> Agente 3
support_workflow = query_processor_agent | content_search_agent | response_generator_agent

print("Workflow LangChain creado correctamente con 3 agentes.")

## 9. Ejemplo práctico solicitado por el enunciado

Consulta simulada: **"¿Cuál es la lección más adecuada para aprender sobre redes neuronales?"**

In [ ]:
example_query = "¿Cuál es la lección más adecuada para aprender sobre redes neuronales?"
example_result = support_workflow.invoke(example_query)

print(example_result["answer"])
print("\n--- JSON estructurado del resultado ---")
print(json.dumps(example_result, indent=2, ensure_ascii=False))

## 10. Pruebas con diferentes escenarios

Se realizan pruebas sobre consultas variadas para comprobar la capacidad de adaptación del sistema:

1. recomendación de lección;
2. error técnico o bug;
3. búsqueda de ejercicio práctico;
4. consulta sobre entrega en GitHub;
5. solicitud con recursos visuales.

In [ ]:
test_queries = [
    "¿Cuál es la lección más adecuada para aprender sobre redes neuronales?",
    "No puedo abrir el ejercicio de fine-tuning de LLMs en Colab, me da error de importación.",
    "Necesito un ejercicio práctico para aprender CNN con imágenes.",
    "¿Cómo debo subir mi notebook al repositorio de GitHub para la entrega?",
    "Quiero una explicación visual sobre Stable Diffusion e image-to-image.",
]

all_results = []
for i, query in enumerate(test_queries, start=1):
    result = support_workflow.invoke(query)
    all_results.append(result)
    print(f"\n================ ESCENARIO {i} ================")
    print(f"Consulta: {query}")
    print(f"Categoría detectada: {result['category']}")
    print(f"Confianza: {result['confidence']}")
    print(result["answer"])

## 11. Validación automática del workflow

Esta celda comprueba que:

- existen tres agentes;
- la base de datos ficticia tiene contenido suficiente;
- las respuestas tienen los campos esperados;
- el sistema responde a distintos tipos de escenarios.

In [ ]:
def validate_workflow(results: List[Dict[str, Any]]) -> None:
    assert len(df_content) >= 10, "La base de datos debe tener al menos 10 registros."
    assert query_processor_agent is not None, "Falta el Agente 1."
    assert content_search_agent is not None, "Falta el Agente 2."
    assert response_generator_agent is not None, "Falta el Agente 3."

    required_keys = {
        "query", "category", "priority", "recommended_course",
        "recommended_lesson", "recommended_exercise", "confidence", "answer", "metadata"
    }
    for result in results:
        missing = required_keys - set(result.keys())
        assert not missing, f"Faltan campos en la respuesta: {missing}"
        assert len(result["answer"]) > 50, "La respuesta generada es demasiado corta."
        assert result["metadata"]["workflow"] == "Agente 1 -> Agente 2 -> Agente 3"

    categories = {result["category"] for result in results}
    assert len(categories) >= 3, "El sistema debería cubrir al menos tres categorías distintas."

    print("Validación completada correctamente: el workflow cumple los requisitos principales.")

validate_workflow(all_results)

## 12. Posible integración real con OpenAI

El workflow anterior funciona sin usar API real. En producción se podría sustituir el Agente 3 por una llamada a `ChatOpenAI`, manteniendo los Agentes 1 y 2 como procesadores y recuperadores de contexto.

Se deja el bloque preparado, pero desactivado por defecto para evitar usar una API key real.

In [ ]:
if USE_OPENAI:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.2,
        api_key=os.environ["OPENAI_API_KEY"],
    )

    # Ejemplo conceptual de uso:
    # prompt = response_prompt_template.format(...)
    # response = llm.invoke(prompt)
else:
    print("Integración real con OpenAI desactivada. El proyecto se ejecuta en modo demo seguro.")

## 13. Conclusión

Este proyecto cumple los requisitos del enunciado porque:

- implementa un flujo con **tres agentes funcionales** y roles diferenciados;
- utiliza **LangChain** mediante `RunnableLambda` y composición LCEL;
- incluye una **base de datos simulada** con cursos, lecciones, ejercicios, imágenes y metadatos;
- maneja consultas variadas: lecciones, ejercicios, bugs, entregas y recursos visuales;
- genera respuestas personalizadas y estructuradas;
- incorpora validaciones automáticas para comprobar el correcto funcionamiento del workflow.

La solución es aplicable a una plataforma educativa real, ya que permite automatizar tickets de soporte, recomendar contenidos y detectar problemas frecuentes de los usuarios.